# Testing coding and testing agents for human eval dataset

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_ollama.chat_models import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import List, TypedDict, Dict, Annotated, Literal, Optional

import common

# initialize code processor
code_processor = common.CodeProcessor()

# sandbox for safe code execution
sandbox: common.SafeCodeSandbox = common.create_safe_test_environment()

# Initialize the Ollama model
llm = ChatOllama(model="qwen2.5-coder:7b")

# Enhanced State with separate message lists for each agent
class AgentState(TypedDict):
    solution: str # The solution code generated by the programmer agent
    tests: str # The tests generated by the test designer agent
    test_results: common.TestExecutionResult # The results after executing the tests

    # Each agent gets its own message list with the add_messages reducer
    programmer_messages: Annotated[List[BaseMessage], add_messages]
    tester_messages: Annotated[List[BaseMessage], add_messages]

# System prompts
PROGRAMMER_SYSTEM_PROMPT = "You are an expert software developer."
TEST_DESIGNER_SYSTEM_PROMPT = "You are an expert program tester."

# Initial Prompts
PROGRAMMER_PROBLEM_PROMPT = """As a programmer, you are required to complete the function. Use a Chain-of-Thought approach to break down the problem, create pseudocode, and then write the code in Python language. Ensure that your code is efficient, readable, and well-commented.  

For example:  
**Input Code Snippet**: 
```python 
{problem_placeholder}
# Add your code here to complete the function 
```  
**Instructions**: 
1. **Understand and Clarify**: Make sure you understand the task. 
2. **Algorithm/Method Selection**: Decide on the most efficient way. 
3. **Pseudocode Creation**: Write down the steps you will follow in pseudocode. 
4. **Code Generation**: Translate your pseudocode into executable Python code."""

TEST_DESIGNER_PROBLEM_PROMPT = """As a tester, your task is to create comprehensive test cases for the incomplete `{function_name}` function. These test cases should encompass Basic, Edge, and Large Scale scenarios to ensure the code's robustness, reliability, and scalability. You will never write the function itself, only the tests.

**Input Code Snippet**: 
```python 
{problem_placeholder}
```

**1. Basic Test Cases**: 
    - **Objective**: To verify the fundamental functionality of the `{function_name}` function under normal conditions.  
**2. Edge Test Cases**: 
    - **Objective**: To evaluate the function's behavior under extreme or unusual conditions.  
**3. Large Scale Test Cases**: 
    - **Objective**: To assess the function's performance and scalability with large data samples.  

**Instructions**: 
    - Use the `unittest` framework for writing the test cases.
    - Implement a comprehensive set of test cases following the guidelines above. 
    - Ensure each test case is well-documented with comments explaining the scenario it covers. 
    - Pay special attention to edge cases as they often reveal hidden bugs. 
    - For large-scale tests, focus on the function's efficiency and performance under heavy loads.
    - Do not implement the function, another expert programmer will do that.
"""

def programmer_agent(state: AgentState) -> AgentState:
    """Programmer agent - only accesses its own message history"""
    
    # Build messages with system prompt + problem + existing programmer conversation
    response = llm.invoke(state['programmer_messages'])
    solution = code_processor.extract_code_block_from_response(response.content)
    
    # Return state updates - the add_messages reducer will handle message accumulation
    return {
        "solution": solution,
        "programmer_messages": [response]
    }

def test_designer_agent(state: AgentState) -> AgentState:
    """Test designer agent - only accesses its own message history"""
    
    response = llm.invoke(state['tester_messages'])
    tests = code_processor.extract_code_block_from_response(response.content)
    
    # Return state updates - the add_messages reducer will handle message accumulation
    return {
        "tests": tests,
        "tester_messages": [response]
    }

def test_executor_agent(state: AgentState) -> AgentState:
    """Test executor agent: This agent runs the tests in a safe sandbox environment. No LLM calls."""

    script = code_processor.build_test_script(state['solution'], state['tests'])
    test_results = sandbox.execute_test_script(script)

    print("\n--- Test Script ---\n")
    print(script)

    print("\n--- Test Execution Results ---\n")
    print(test_results.execution_category)
    print("Number of Tests Passed:", test_results.tests_passed)
    print("Number of Tests Failed:", test_results.tests_failed)
    print("Number of Tests Errors:", test_results.tests_errors)

    print("\n--- Test Summary ---\n")
    print(test_results.test_summary)

    if test_results.tests_failed > 0:
        print("\n--- Failed Tests Details ---\n")
        for failed_test in test_results.test_details.failed_test_details:
            print("Name: ", failed_test.name)
            print("Description: ", failed_test.description)
            print("Details: ", failed_test.details)

    if test_results.tests_passed > 0:
        print("\n--- Passed Tests Details ---\n")
        for passed_test in test_results.test_details.passed_test_details:
            print("Name: ", passed_test.name)
            print("Description: ", passed_test.description)

     
    if test_results.tests_errors > 0:
        print("\n--- Error Tests Details ---\n")
        for error_test in test_results.test_details.error_test_details:
            print("Name: ", error_test.name)
            print("Description: ", error_test.description)
            print("Details: ", error_test.details)

    return {
        "test_results": test_results, 
        "programmer_messages": [HumanMessage(content="Your code did not pass all the tests.")]
    }

# def route_by_test_results(state: AgentState) -> Literal["programmer", "test_designer"]:
#     if state['test_results'].execution_category ==

# 3. Build and run the graph
graph = StateGraph(AgentState)
graph.add_node("programmer", programmer_agent)
graph.add_node("test_designer", test_designer_agent)
graph.add_node("test_executor", test_executor_agent)

graph.add_edge(START, "programmer")
graph.add_edge("programmer", "test_designer")
graph.add_edge("test_designer", 'test_executor')
graph.add_edge("test_executor", END)

# Compile the graph
agent_coder = graph.compile()

# Example problem for testing
sample_problem = '''
from typing import List    
    def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """'''

print("Starting Multi-Agent Code Generation and Testing System")
print("=" * 60)

# Initialize state - much cleaner without manual memory management
initial_state = {
    "solution": "",
    "tests": "",
    "test_results": None,
    "programmer_messages": [SystemMessage(content=PROGRAMMER_SYSTEM_PROMPT)] + [HumanMessage(content=PROGRAMMER_PROBLEM_PROMPT.format(problem_placeholder=sample_problem))],  # Will accumulate programmer's conversation
    "tester_messages": [SystemMessage(content=TEST_DESIGNER_SYSTEM_PROMPT)] + [HumanMessage(content=TEST_DESIGNER_PROBLEM_PROMPT.format(
        function_name=code_processor.extract_function_name_from_problem(sample_problem), 
        problem_placeholder=sample_problem
    ))],  # Will accumulate tester's conversation
}

# Run the multi-agent workflow
result = agent_coder.invoke(initial_state)

print("=" * 60)
# print("\n--- Generated Solution ---\n")
# print(result['solution'])

# print("\n--- Generated Tests ---\n")
# print(result['tests'])

Starting Multi-Agent Code Generation and Testing System

--- Test Script ---

from typing import List
import time
import unittest

# Programmer Code
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    # Sort the list
    numbers.sort()
    
    # Iterate through the sorted list and check adjacent elements
    for i in range(len(numbers) - 1):
        if abs(numbers[i] - numbers[i+1]) < threshold:
            return True
    
    return False


# Tester Code
class TestHasCloseElements(unittest.TestCase):
    
    def test_basic_cases(self):
        """Test basic functionality with normal inputs."""
        self.assertFalse(has_close_elements([1.0, 2.0, 3.0], 0.5))
        self.assertTrue(has_close_elements([1.0, 2.8, 